In [3]:
import numpy as np
from scipy.stats import norm
import pandas as pd

class BinomialOptionPricer:
    def __init__(self, S0, K, T, r, sigma, N=100):
        """
        Initialize the Binomial Option Pricer.
        
        Parameters:
        -----------
        S0 : float
            Initial stock price
        K : float
            Strike price
        T : float
            Time to maturity (in years)
        r : float
            Risk-free rate
        sigma : float
            Volatility
        N : int
            Number of time steps
        """
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.N = N
        self.dt = T/N
        
        # Calculate tree parameters
        self.u = np.exp(sigma * np.sqrt(self.dt))
        self.d = 1/self.u
        self.p = (np.exp(r*self.dt) - self.d)/(self.u - self.d)
        
    def price_american_option(self, option_type='put'):
        """Price American option using binomial model."""
        # Initialize arrays for efficiency
        S = np.zeros(self.N + 1)
        V = np.zeros(self.N + 1)
        
        # Terminal stock prices
        for i in range(self.N + 1):
            S[i] = self.S0 * (self.u ** (self.N - i)) * (self.d ** i)
        
        # Terminal option values
        for i in range(self.N + 1):
            if option_type == 'put':
                V[i] = max(0, self.K - S[i])
            else:
                V[i] = max(0, S[i] - self.K)
        
        # Backward induction
        for j in range(self.N-1, -1, -1):
            for i in range(j + 1):
                spot = self.S0 * (self.u ** (j - i)) * (self.d ** i)
                hold_value = np.exp(-self.r * self.dt) * (
                    self.p * V[i] + (1 - self.p) * V[i + 1]
                )
                
                if option_type == 'put':
                    exercise_value = max(0, self.K - spot)
                else:
                    exercise_value = max(0, spot - self.K)
                    
                V[i] = max(hold_value, exercise_value)
        
        return V[0]
    
    def black_scholes_price(self, option_type='put'):
        """Calculate Black-Scholes price for comparison."""
        d1 = (np.log(self.S0/self.K) + (self.r + 0.5*self.sigma**2)*self.T) / (self.sigma*np.sqrt(self.T))
        d2 = d1 - self.sigma*np.sqrt(self.T)
        
        if option_type == 'put':
            return self.K*np.exp(-self.r*self.T)*norm.cdf(-d2) - self.S0*norm.cdf(-d1)
        else:
            return self.S0*norm.cdf(d1) - self.K*np.exp(-self.r*self.T)*norm.cdf(d2)
    
    def calculate_greeks(self, option_type='put'):
        """Calculate option Greeks."""
        delta_S = self.S0 * 0.01  # 1% change in stock price
        delta_t = self.T / 365    # 1 day change
        delta_sigma = 0.01        # 1% change in volatility
        
        # Base price
        base_price = self.price_american_option(option_type)
        
        # Delta
        up_price = BinomialOptionPricer(self.S0 + delta_S, self.K, self.T, self.r, self.sigma, self.N).price_american_option(option_type)
        delta = (up_price - base_price) / delta_S
        
        # Gamma
        down_price = BinomialOptionPricer(self.S0 - delta_S, self.K, self.T, self.r, self.sigma, self.N).price_american_option(option_type)
        gamma = (up_price - 2*base_price + down_price) / (delta_S**2)
        
        # Theta
        later_price = BinomialOptionPricer(self.S0, self.K, self.T - delta_t, self.r, self.sigma, self.N).price_american_option(option_type)
        theta = (later_price - base_price) / delta_t
        
        # Vega
        vol_up_price = BinomialOptionPricer(self.S0, self.K, self.T, self.r, self.sigma + delta_sigma, self.N).price_american_option(option_type)
        vega = (vol_up_price - base_price) / delta_sigma
        
        return {
            'delta': delta,
            'gamma': gamma,
            'theta': theta,
            'vega': vega
        }
    
    def analyze_option(self, option_type='put'):
        """Comprehensive option analysis."""
        # Price both American and European (Black-Scholes) versions
        american_price = self.price_american_option(option_type)
        bs_price = self.black_scholes_price(option_type)
        
        # Calculate Greeks
        greeks = self.calculate_greeks(option_type)
        
        # Calculate early exercise premium
        early_exercise_premium = american_price - bs_price
        
        # Calculate implied volatility metrics
        moneyness = self.S0/self.K
        
        # Calculate break-even price
        if option_type == 'put':
            break_even = self.K - american_price
        else:
            break_even = self.K + american_price
            
        return {
            'american_price': american_price,
            'black_scholes_price': bs_price,
            'early_exercise_premium': early_exercise_premium,
            'break_even': break_even,
            'moneyness': moneyness,
            'greeks': greeks,
            'time_value': american_price - max(0, self.K - self.S0) if option_type == 'put' 
                         else american_price - max(0, self.S0 - self.K)
        }

def generate_pricing_table(S0, K_range, T, r, sigma):
    """Generate pricing analysis for multiple strikes."""
    results = []
    
    for K in K_range:
        pricer = BinomialOptionPricer(S0, K, T, r, sigma)
        put_analysis = pricer.analyze_option('put')
        call_analysis = pricer.analyze_option('call')
        
        results.append({
            'Strike': K,
            'Put Price': put_analysis['american_price'],
            'Put Delta': put_analysis['greeks']['delta'],
            'Put Theta': put_analysis['greeks']['theta'],
            'Call Price': call_analysis['american_price'],
            'Call Delta': call_analysis['greeks']['delta'],
            'Call Theta': call_analysis['greeks']['theta'],
            'Put Early Exercise Premium': put_analysis['early_exercise_premium'],
            'Call Early Exercise Premium': call_analysis['early_exercise_premium']
        })
    
    return pd.DataFrame(results).round(4)

# Example usage
if __name__ == "__main__":
    # Test parameters
    S0 = 100
    K = 100
    T = 1.0
    r = 0.05
    sigma = 0.2
    
    # Create pricer instance
    pricer = BinomialOptionPricer(S0, K, T, r, sigma)
    
    # Analyze put option
    put_analysis = pricer.analyze_option('put')
    
    print("American Put Option Analysis")
    print("-" * 50)
    print(f"Price: ${put_analysis['american_price']:.4f}")
    print(f"Black-Scholes Price: ${put_analysis['black_scholes_price']:.4f}")
    print(f"Early Exercise Premium: ${put_analysis['early_exercise_premium']:.4f}")
    print(f"Break-even Price: ${put_analysis['break_even']:.4f}")
    print(f"Time Value: ${put_analysis['time_value']:.4f}")
    print("\nGreeks:")
    for greek, value in put_analysis['greeks'].items():
        print(f"{greek.capitalize()}: {value:.4f}")
    
    # Generate pricing table for different strikes
    strikes = np.linspace(90, 110, 5)
    pricing_table = generate_pricing_table(S0, strikes, T, r, sigma)
    
    print("\nPricing Analysis for Different Strikes:")
    print(pricing_table)

American Put Option Analysis
--------------------------------------------------
Price: $6.0824
Black-Scholes Price: $5.5735
Early Exercise Premium: $0.5088
Break-even Price: $93.9176
Time Value: $6.0824

Greeks:
Delta: -0.3858
Gamma: 0.0548
Theta: -2.2372
Vega: 37.4841

Pricing Analysis for Different Strikes:
   Strike  Put Price  Put Delta  Put Theta  Call Price  Call Delta  \
0    90.0     2.4831    -0.2102    -1.9423     16.7105      0.8025   
1    95.0     4.0202    -0.2859    -2.1520     13.3556      0.7424   
2   100.0     6.0824    -0.3858    -2.2372     10.4306      0.6739   
3   105.0     8.7476    -0.5324    -2.0401      8.0262      0.5296   
4   110.0    11.9812    -0.6479    -1.6746      6.0532      0.4408   

   Call Theta  Put Early Exercise Premium  Call Early Exercise Premium  
0     -5.9746                      0.1730                       0.0110  
1     -6.2602                      0.3070                       0.0092  
2     -6.4069                      0.5088        